# TransReID Experiment Suite

Pipeline này chạy thí nghiệm cho report:

- Market1501 và Occluded-DukeMTMC.
- Fair comparison: `baseline`, `sem_align`, `local_reliability`, `sem_align_reliability`.
- Tất cả method dùng cùng batch size, learning rate, warmup/eval/checkpoint cadence.
- Train, evaluate mAP/Rank, đo latency, sinh CSV/JSON/plot và zip toàn bộ logs + checkpoints.

Để smoke test nhanh, đặt `MAX_EPOCHS_OVERRIDE = 1` hoặc `2`.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

CONTENT_ROOT = Path('/content')
REPO_ROOT = CONTENT_ROOT / 'transreid_colab'
ZIP_PATH = CONTENT_ROOT / 'transreid_colab.zip'

if not REPO_ROOT.exists():
    if ZIP_PATH.exists():
        shutil.unpack_archive(str(ZIP_PATH), str(CONTENT_ROOT))
    else:
        raise FileNotFoundError('Không thấy /content/transreid_colab hoặc /content/transreid_colab.zip')

os.chdir(REPO_ROOT)
print('Repo:', Path.cwd())

In [ ]:
%%shell
set -euo pipefail
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m pip install gdown transformers accelerate tqdm pandas matplotlib seaborn

mkdir -p /content/pretrained /content/data /content/transreid_artifacts
if [ ! -f /content/pretrained/jx_vit_base_p16_224-80ecf9dd.pth ]; then
  wget -O /content/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
    https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
fi

In [ ]:
from pathlib import Path
import json
import re
import shlex
import shutil
import subprocess
import sys
import time

DATA_ROOT = Path('/content/data')
PRETRAIN_PATH = Path('/content/pretrained/jx_vit_base_p16_224-80ecf9dd.pth')
RUN_ROOT = Path('runs')
EVAL_ROOT = Path('eval_runs')
ARTIFACT_ROOT = Path('/content/transreid_artifacts')
TRAIN_LOG_DIR = ARTIFACT_ROOT / 'train_logs'
EVAL_LOG_DIR = ARTIFACT_ROOT / 'eval_logs'
LATENCY_DIR = ARTIFACT_ROOT / 'latency'
PLOT_DIR = ARTIFACT_ROOT / 'plots'
for path in [RUN_ROOT, EVAL_ROOT, ARTIFACT_ROOT, TRAIN_LOG_DIR, EVAL_LOG_DIR, LATENCY_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE_ID = "'0'"       # yacs cần string, không dùng "0"
NUM_WORKERS = 8
TEST_BATCH_SIZE = 256

# Parallel training. Giữ 1 nếu Colab chỉ có 1 GPU hoặc muốn chạy tuần tự.
# Ví dụ 2 GPU: TRAIN_PARALLEL_JOBS = 2 và TRAIN_GPU_IDS = ['0', '1'].
# Nếu số jobs > số GPU thì các jobs sẽ share GPU theo round-robin, dễ OOM với batch 64.
TRAIN_PARALLEL_JOBS = 1
TRAIN_GPU_IDS = ['0']
PARALLEL_STATUS_SECONDS = 60

PREPARE_MARKET1501 = True
PREPARE_OCC_DUKE = True
PREPARE_SEMANTIC_MAPS = True
RUN_TRAIN = True
RUN_EVAL = True
RUN_LATENCY = True
DOWNLOAD_ZIP = False

MAX_EPOCHS_OVERRIDE = None
EVAL_PERIOD_OVERRIDE = None
CHECKPOINT_PERIOD_OVERRIDE = None

SEMANTIC_PRESET = 'lip6'
SEMANTIC_BATCH_SIZE = 64

LATENCY_BATCH_SIZE = 1
LATENCY_WARMUP = 30
LATENCY_ITERS = 100
LATENCY_USE_AMP = False

print('DATA_ROOT:', DATA_ROOT)
print('PRETRAIN_PATH exists:', PRETRAIN_PATH.exists())

In [ ]:
MARKET_URL = 'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ&usp=sharing'


def run_cmd(cmd, log_path=None, cwd=None):
    display = cmd if isinstance(cmd, str) else ' '.join(shlex.quote(str(x)) for x in cmd)
    print('\n$ ' + display)
    log_handle = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_path.open('w', encoding='utf-8')
        log_handle.write('$ ' + display + '\n')
    try:
        proc = subprocess.Popen(
            cmd if isinstance(cmd, str) else [str(x) for x in cmd],
            cwd=str(cwd) if cwd else None,
            shell=isinstance(cmd, str),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            if log_handle:
                log_handle.write(line)
        rc = proc.wait()
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
    finally:
        if log_handle:
            log_handle.close()


def prepare_market1501():
    market_root = DATA_ROOT / 'market1501'
    if (market_root / 'bounding_box_train').exists() and (market_root / 'query').exists() and (market_root / 'bounding_box_test').exists():
        print('Market1501 already prepared:', market_root)
        return
    archive = DATA_ROOT / 'Market-1501-v15.09.15.zip'
    run_cmd(['gdown', '--fuzzy', MARKET_URL, '-O', archive])
    tmp_extract = DATA_ROOT / '_market_extract'
    if tmp_extract.exists():
        shutil.rmtree(tmp_extract)
    shutil.unpack_archive(str(archive), str(tmp_extract))
    extracted = tmp_extract / 'Market-1501-v15.09.15'
    if market_root.exists():
        shutil.rmtree(market_root)
    shutil.move(str(extracted), str(market_root))
    shutil.rmtree(tmp_extract)
    print('Market1501 prepared:', market_root)


def prepare_occ_duke():
    occ_root = DATA_ROOT / 'Occluded_Duke'
    if (occ_root / 'bounding_box_train').exists() and (occ_root / 'query').exists() and (occ_root / 'bounding_box_test').exists():
        print('Occluded-Duke already prepared:', occ_root)
        return
    run_cmd(['bash', 'tools/colab_prepare_occ_duke.sh', str(occ_root), str(DATA_ROOT / '.cache/occluded_duke_prepare')])


if PREPARE_MARKET1501:
    prepare_market1501()
if PREPARE_OCC_DUKE:
    prepare_occ_duke()

In [ ]:
SEMANTIC_DATASETS = [
    ('market1501', DATA_ROOT / 'market1501'),
    ('occ_duke', DATA_ROOT / 'Occluded_Duke'),
]


def semantic_maps_ready(dataset_root):
    root = dataset_root / 'semantic_groups'
    return (root / 'bounding_box_train').exists() and any((root / 'bounding_box_train').glob('*.png'))


if PREPARE_SEMANTIC_MAPS:
    for dataset_name, dataset_root in SEMANTIC_DATASETS:
        if not dataset_root.exists():
            print('Skip semantic maps, dataset missing:', dataset_root)
            continue
        if semantic_maps_ready(dataset_root):
            print('Semantic maps already exist:', dataset_root / 'semantic_groups')
            continue
        run_cmd([
            sys.executable, 'tools/prepare_semantic_maps.py',
            '--dataset-root', dataset_root,
            '--output-root', dataset_root / 'semantic_groups',
            '--raw-output-root', dataset_root / 'raw_parsing',
            '--preset', SEMANTIC_PRESET,
            '--subdirs', 'bounding_box_train', 'query', 'bounding_box_test',
            '--batch-size', SEMANTIC_BATCH_SIZE,
            '--device', 'cuda',
        ])

In [ ]:
EXPERIMENTS = [
    {'name': 'market_baseline', 'dataset': 'market1501', 'config': 'configs/Market/vit_transreid_stride_fair.yml', 'output_dir': 'runs/fair_market_baseline', 'eval_dir': 'eval_runs/fair_market_baseline', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 751, 'camera_num': 6, 'view_num': 0, 'enabled': True},
    {'name': 'market_sem_align', 'dataset': 'market1501', 'config': 'configs/Market/vit_transreid_stride_sem_align.yml', 'output_dir': 'runs/fair_market_sem_align', 'eval_dir': 'eval_runs/fair_market_sem_align', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 751, 'camera_num': 6, 'view_num': 0, 'enabled': True},
    {'name': 'market_local_reliability', 'dataset': 'market1501', 'config': 'configs/Market/vit_transreid_stride_local_reliability.yml', 'output_dir': 'runs/fair_market_local_reliability', 'eval_dir': 'eval_runs/fair_market_local_reliability', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 751, 'camera_num': 6, 'view_num': 0, 'enabled': True},
    {'name': 'market_sem_align_reliability', 'dataset': 'market1501', 'config': 'configs/Market/vit_transreid_stride_sem_align_reliability.yml', 'output_dir': 'runs/fair_market_sem_align_reliability', 'eval_dir': 'eval_runs/fair_market_sem_align_reliability', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 751, 'camera_num': 6, 'view_num': 0, 'enabled': True},
    {'name': 'occ_duke_baseline', 'dataset': 'occ_duke', 'config': 'configs/OCC_Duke/vit_transreid_stride_fair.yml', 'output_dir': 'runs/fair_occ_duke_baseline', 'eval_dir': 'eval_runs/fair_occ_duke_baseline', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 702, 'camera_num': 8, 'view_num': 0, 'enabled': True},
    {'name': 'occ_duke_sem_align', 'dataset': 'occ_duke', 'config': 'configs/OCC_Duke/vit_transreid_stride_sem_align.yml', 'output_dir': 'runs/fair_occ_duke_sem_align', 'eval_dir': 'eval_runs/fair_occ_duke_sem_align', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 702, 'camera_num': 8, 'view_num': 0, 'enabled': True},
    {'name': 'occ_duke_local_reliability', 'dataset': 'occ_duke', 'config': 'configs/OCC_Duke/vit_transreid_stride_local_reliability.yml', 'output_dir': 'runs/fair_occ_duke_local_reliability', 'eval_dir': 'eval_runs/fair_occ_duke_local_reliability', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 702, 'camera_num': 8, 'view_num': 0, 'enabled': True},
    {'name': 'occ_duke_sem_align_reliability', 'dataset': 'occ_duke', 'config': 'configs/OCC_Duke/vit_transreid_stride_sem_align_reliability.yml', 'output_dir': 'runs/fair_occ_duke_sem_align_reliability', 'eval_dir': 'eval_runs/fair_occ_duke_sem_align_reliability', 'batch_size': 64, 'base_lr': 0.008, 'num_workers': 8, 'num_classes': 702, 'camera_num': 8, 'view_num': 0, 'enabled': True},
]

enabled_experiments = [exp for exp in EXPERIMENTS if exp.get('enabled', True)]
print(json.dumps(enabled_experiments, indent=2))

In [ ]:
def checkpoint_epoch(path):
    match = re.search(r'_(\d+)\.pth$', path.name)
    return int(match.group(1)) if match else -1


def find_checkpoint(output_dir):
    output_dir = Path(output_dir)
    best = output_dir / 'transformer_best.pth'
    if best.exists():
        return best
    candidates = sorted(output_dir.glob('transformer_*.pth'), key=checkpoint_epoch)
    if candidates:
        return candidates[-1]
    latest = output_dir / 'latest_model.pth'
    if latest.exists():
        return latest
    return None


def yacs_device_id(device_id):
    text = str(device_id)
    if text.startswith("'") or text.startswith('"') or text.startswith('('):
        return text
    return f"'{text}'"


def train_command(exp, device_id=None):
    device_id = DEVICE_ID if device_id is None else yacs_device_id(device_id)
    cmd = [
        sys.executable, 'train.py',
        '--config_file', exp['config'],
        'MODEL.DEVICE_ID', device_id,
        'MODEL.PRETRAIN_CHOICE', 'imagenet',
        'MODEL.PRETRAIN_PATH', str(PRETRAIN_PATH),
        'DATASETS.ROOT_DIR', str(DATA_ROOT) + '/',
        'OUTPUT_DIR', exp['output_dir'],
        'SOLVER.IMS_PER_BATCH', str(exp['batch_size']),
        'SOLVER.BASE_LR', str(exp['base_lr']),
        'DATALOADER.NUM_WORKERS', str(exp.get('num_workers', NUM_WORKERS)),
        'TEST.IMS_PER_BATCH', str(TEST_BATCH_SIZE),
    ]
    if MAX_EPOCHS_OVERRIDE is not None:
        cmd.extend(['SOLVER.MAX_EPOCHS', str(MAX_EPOCHS_OVERRIDE)])
    if EVAL_PERIOD_OVERRIDE is not None:
        cmd.extend(['SOLVER.EVAL_PERIOD', str(EVAL_PERIOD_OVERRIDE)])
    if CHECKPOINT_PERIOD_OVERRIDE is not None:
        cmd.extend(['SOLVER.CHECKPOINT_PERIOD', str(CHECKPOINT_PERIOD_OVERRIDE)])
    return cmd


def eval_command(exp, checkpoint, device_id=None):
    device_id = DEVICE_ID if device_id is None else yacs_device_id(device_id)
    return [
        sys.executable, 'test.py',
        '--config_file', exp['config'],
        'MODEL.DEVICE_ID', device_id,
        'MODEL.PRETRAIN_CHOICE', 'none',
        'DATASETS.ROOT_DIR', str(DATA_ROOT) + '/',
        'OUTPUT_DIR', exp['eval_dir'],
        'TEST.WEIGHT', str(checkpoint),
        'TEST.IMS_PER_BATCH', str(TEST_BATCH_SIZE),
    ]


def make_train_jobs(experiments):
    gpu_ids = TRAIN_GPU_IDS or [DEVICE_ID]
    jobs = []
    for index, exp in enumerate(experiments):
        gpu_id = gpu_ids[index % len(gpu_ids)]
        jobs.append({
            'name': exp['name'],
            'gpu_id': gpu_id,
            'cmd': train_command(exp, gpu_id),
            'log_path': TRAIN_LOG_DIR / f"{exp['name']}_train.log",
        })
    return jobs


def run_parallel_train_batch(jobs):
    running = []
    for job in jobs:
        log_path = Path(job['log_path'])
        log_path.parent.mkdir(parents=True, exist_ok=True)
        display = ' '.join(shlex.quote(str(x)) for x in job['cmd'])
        log_handle = log_path.open('w', encoding='utf-8')
        log_handle.write('$ ' + display + '\n')
        log_handle.flush()
        proc = subprocess.Popen(
            [str(x) for x in job['cmd']],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
        )
        running.append({**job, 'proc': proc, 'log_handle': log_handle, 'start_time': time.time()})
        print(f"Started {job['name']} on GPU {job['gpu_id']} pid={proc.pid} log={log_path}")

    failures = []
    last_status = time.time()
    while running:
        now = time.time()
        if now - last_status >= PARALLEL_STATUS_SECONDS:
            names = ', '.join(f"{item['name']}(gpu={item['gpu_id']}, pid={item['proc'].pid})" for item in running)
            print(f"Still running: {names}")
            last_status = now

        for item in list(running):
            rc = item['proc'].poll()
            if rc is None:
                continue
            item['log_handle'].close()
            elapsed_min = (time.time() - item['start_time']) / 60.0
            running.remove(item)
            if rc == 0:
                print(f"Finished {item['name']} in {elapsed_min:.1f} min")
            else:
                failures.append(item)
                print(f"FAILED {item['name']} rc={rc} after {elapsed_min:.1f} min. See log: {item['log_path']}")

        if running:
            time.sleep(5)

    if failures:
        failed = ', '.join(item['name'] for item in failures)
        raise RuntimeError(f'Parallel training failed: {failed}')


def run_training_stage(experiments):
    if not RUN_TRAIN:
        print('RUN_TRAIN=False, skip training stage')
        return
    parallel_jobs = max(1, int(TRAIN_PARALLEL_JOBS))
    gpu_ids = TRAIN_GPU_IDS or [DEVICE_ID]
    if parallel_jobs > len(gpu_ids):
        print(f'Warning: TRAIN_PARALLEL_JOBS={parallel_jobs} > len(TRAIN_GPU_IDS)={len(gpu_ids)}. Jobs will share GPUs; batch 64 may OOM.')
    jobs = make_train_jobs(experiments)
    for start in range(0, len(jobs), parallel_jobs):
        batch = jobs[start:start + parallel_jobs]
        print('\n' + '=' * 100)
        print('Training batch:', ', '.join(job['name'] for job in batch))
        print('=' * 100)
        if parallel_jobs == 1:
            job = batch[0]
            run_cmd(job['cmd'], job['log_path'])
        else:
            run_parallel_train_batch(batch)


checkpoint_manifest = {}
run_training_stage(enabled_experiments)

for exp in enabled_experiments:
    print('\n' + '=' * 100)
    print('Collect/evaluate:', exp['name'])
    print('=' * 100)
    ckpt = find_checkpoint(exp['output_dir'])
    if ckpt is None:
        print('No checkpoint found:', exp['output_dir'])
        checkpoint_manifest[exp['name']] = None
        continue
    checkpoint_manifest[exp['name']] = str(ckpt)
    print('Checkpoint:', ckpt)
    if RUN_EVAL:
        run_cmd(eval_command(exp, ckpt), EVAL_LOG_DIR / f"{exp['name']}_eval.log")

(ARTIFACT_ROOT / 'checkpoint_manifest.json').write_text(json.dumps(checkpoint_manifest, indent=2), encoding='utf-8')
checkpoint_manifest

In [ ]:
import pandas as pd


def parse_eval_log(log_path):
    text = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    maps = [float(x) for x in re.findall(r'mAP:\s*([0-9.]+)%', text)]
    ranks = {int(r): float(v) for r, v in re.findall(r'Rank-\s*(\d+)\s*:\s*([0-9.]+)%', text)}
    return {'mAP': maps[-1] if maps else None, 'rank1': ranks.get(1), 'rank5': ranks.get(5), 'rank10': ranks.get(10)}


rows = []
for exp in enabled_experiments:
    row = {'name': exp['name'], 'dataset': exp['dataset'], 'config': exp['config'], 'output_dir': exp['output_dir'], 'checkpoint': checkpoint_manifest.get(exp['name'])}
    eval_log = EVAL_LOG_DIR / f"{exp['name']}_eval.log"
    train_log = TRAIN_LOG_DIR / f"{exp['name']}_train.log"
    if eval_log.exists():
        row.update(parse_eval_log(eval_log))
        row['eval_log'] = str(eval_log)
    elif train_log.exists():
        row.update(parse_eval_log(train_log))
        row['eval_log'] = ''
    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(ARTIFACT_ROOT / 'metrics_summary.csv', index=False)
metrics_df.to_json(ARTIFACT_ROOT / 'metrics_summary.json', orient='records', indent=2)
metrics_df

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from config import cfg
from model import make_model


def set_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_cfg_for_latency(config_path):
    c = cfg.clone()
    c.merge_from_file(config_path)
    c.MODEL.PRETRAIN_CHOICE = 'none'
    c.freeze()
    return c


def load_checkpoint_partial(model, checkpoint_path):
    checkpoint = torch.load(str(checkpoint_path), map_location='cpu')
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        checkpoint = checkpoint['model']
    model_state = model.state_dict()
    filtered, skipped = {}, []
    for key, value in checkpoint.items():
        clean_key = key.replace('module.', '')
        if clean_key in model_state and model_state[clean_key].shape == value.shape:
            filtered[clean_key] = value
        else:
            skipped.append(clean_key)
    model.load_state_dict(filtered, strict=False)
    return len(filtered), len(skipped)


def benchmark_latency(exp, checkpoint_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    c = load_cfg_for_latency(exp['config'])
    model = make_model(c, num_class=exp['num_classes'], camera_num=exp['camera_num'], view_num=exp['view_num'])
    loaded, skipped = load_checkpoint_partial(model, checkpoint_path)
    model.to(device).eval()
    h, w = c.INPUT.SIZE_TEST
    images = torch.randn(LATENCY_BATCH_SIZE, 3, h, w, device=device)
    camids = torch.zeros(LATENCY_BATCH_SIZE, dtype=torch.long, device=device)
    viewids = torch.zeros(LATENCY_BATCH_SIZE, dtype=torch.long, device=device)
    autocast_device = 'cuda' if device.type == 'cuda' else 'cpu'
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    timings_ms = []
    with torch.inference_mode():
        for _ in range(LATENCY_WARMUP):
            with torch.autocast(device_type=autocast_device, enabled=LATENCY_USE_AMP):
                _ = model(images, cam_label=camids, view_label=viewids)
        if device.type == 'cuda':
            torch.cuda.synchronize()
            for _ in range(LATENCY_ITERS):
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
                with torch.autocast(device_type=autocast_device, enabled=LATENCY_USE_AMP):
                    _ = model(images, cam_label=camids, view_label=viewids)
                end.record()
                torch.cuda.synchronize()
                timings_ms.append(start.elapsed_time(end))
        else:
            for _ in range(LATENCY_ITERS):
                t0 = time.perf_counter()
                with torch.autocast(device_type=autocast_device, enabled=LATENCY_USE_AMP):
                    _ = model(images, cam_label=camids, view_label=viewids)
                timings_ms.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(timings_ms, dtype=np.float64)
    raw_path = LATENCY_DIR / f"{exp['name']}_latency_raw.npy"
    np.save(raw_path, arr)
    return {
        'name': exp['name'], 'dataset': exp['dataset'], 'config': exp['config'],
        'checkpoint': str(checkpoint_path), 'device': str(device), 'batch_size': LATENCY_BATCH_SIZE,
        'warmup': LATENCY_WARMUP, 'iters': LATENCY_ITERS, 'amp': LATENCY_USE_AMP,
        'loaded_tensors': loaded, 'skipped_tensors': skipped,
        'mean_ms': float(arr.mean()), 'std_ms': float(arr.std()),
        'p50_ms': float(np.percentile(arr, 50)), 'p95_ms': float(np.percentile(arr, 95)), 'p99_ms': float(np.percentile(arr, 99)),
        'min_ms': float(arr.min()), 'max_ms': float(arr.max()), 'fps': float(1000.0 / arr.mean() * LATENCY_BATCH_SIZE),
        'raw_path': str(raw_path),
    }


latency_rows = []
if RUN_LATENCY:
    set_seed()
    torch.backends.cudnn.benchmark = True
    for exp in enabled_experiments:
        checkpoint = checkpoint_manifest.get(exp['name'])
        if not checkpoint or not Path(checkpoint).exists():
            print('Skip latency, missing checkpoint:', exp['name'])
            continue
        print('\nLatency:', exp['name'])
        latency_rows.append(benchmark_latency(exp, Path(checkpoint)))

latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv(ARTIFACT_ROOT / 'latency_summary.csv', index=False)
latency_df.to_json(ARTIFACT_ROOT / 'latency_summary.json', orient='records', indent=2)
latency_df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if (ARTIFACT_ROOT / 'metrics_summary.csv').exists():
    df = pd.read_csv(ARTIFACT_ROOT / 'metrics_summary.csv')
    for metric in ['mAP', 'rank1', 'rank5', 'rank10']:
        if metric in df.columns and df[metric].notna().any():
            ax = df.plot(kind='bar', x='name', y=metric, figsize=(12, 5), legend=False, title=metric)
            ax.set_ylabel(metric + ' (%)')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            out = PLOT_DIR / f'{metric}_bar.png'
            plt.savefig(out, dpi=160)
            plt.close()
            print('Saved', out)

if (ARTIFACT_ROOT / 'latency_summary.csv').exists():
    ldf = pd.read_csv(ARTIFACT_ROOT / 'latency_summary.csv')
    if not ldf.empty:
        ax = ldf.plot(kind='bar', x='name', y='mean_ms', figsize=(12, 5), legend=False, title='Mean Inference Latency')
        ax.set_ylabel('ms / batch')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        out = PLOT_DIR / 'latency_mean_ms_bar.png'
        plt.savefig(out, dpi=160)
        plt.close()
        print('Saved', out)

In [ ]:
target_last_epoch = int(MAX_EPOCHS_OVERRIDE or 120)
checkpoint_keep_names = {'transformer_best.pth', f'transformer_{target_last_epoch}.pth'}


def checkpoint_epoch_from_name(name):
    match = re.search(r'_(\d+)\.pth$', name)
    return int(match.group(1)) if match else -1


def runs_checkpoint_ignore(dir_path, names):
    keep_names = set(checkpoint_keep_names)
    epoch_checkpoints = [name for name in names if re.match(r'transformer_\d+\.pth$', name)]
    target_last_name = f'transformer_{target_last_epoch}.pth'
    if target_last_name not in names and epoch_checkpoints:
        keep_names.add(max(epoch_checkpoints, key=checkpoint_epoch_from_name))

    ignored = []
    for name in names:
        path = Path(dir_path) / name
        if path.is_file() and path.suffix == '.pth' and name not in keep_names:
            ignored.append(name)
    return ignored


def copy_artifact_tree(src, dst, ignore=None):
    if not src.exists():
        return
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=ignore)


manifest = {
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'repo_root': str(Path.cwd()),
    'data_root': str(DATA_ROOT),
    'pretrain_path': str(PRETRAIN_PATH),
    'experiments': enabled_experiments,
    'checkpoints': checkpoint_manifest,
    'checkpoint_keep_names': sorted(checkpoint_keep_names),
    'max_epochs_override': MAX_EPOCHS_OVERRIDE,
    'latency': {'batch_size': LATENCY_BATCH_SIZE, 'warmup': LATENCY_WARMUP, 'iters': LATENCY_ITERS, 'amp': LATENCY_USE_AMP},
}
(ARTIFACT_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

for src, dst, ignore in [
    (Path('runs'), ARTIFACT_ROOT / 'runs', runs_checkpoint_ignore),
    (Path('eval_runs'), ARTIFACT_ROOT / 'eval_runs', None),
    (Path('configs'), ARTIFACT_ROOT / 'configs_snapshot', None),
]:
    copy_artifact_tree(src, dst, ignore=ignore)

print('Checkpoint files kept in runs snapshot:', ', '.join(sorted(checkpoint_keep_names)))

zip_base = Path('/content') / f"transreid_report_artifacts_{time.strftime('%Y%m%d_%H%M%S')}"
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(ARTIFACT_ROOT.parent), base_dir=ARTIFACT_ROOT.name)
print('Artifact zip:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / (1024 * 1024))

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(zip_path)